# Master Dataset Construction ( Leakage-Free Pipeline )

## 1. Overview

This notebook constructs the final multimodal breast cancer Recurrence-Free Survival (RFS) dataset by combining clinical, genomic, pathology, and follow-up data.

The dataset uses the corrected patient splits and the authoritative RFS endpoint. Patient IDs are used to match the different data sources and ensure that each patient's information belongs to the correct record.

## 2. Data Sources

The master dataset combines four main data sources:

- **Clinical data:** Preprocessed clinical features from the leakage-free clinical pipeline.
- **Genomic data:** The 200 selected gene features obtained from the leakage-free feature selection pipeline.
- **Pathology data:** Existing pathology feature arrays in `.npy` format, referenced through their patient IDs and file UUIDs.
- **RFS endpoint:** The authoritative survival time and event label for each patient.

The corrected train, validation, and test patient IDs are also included to preserve the existing data split assignments.

## 3. Main Processing Steps

1. **Load the data:** Load the corrected patient splits and the preprocessed clinical, genomic, pathology, and RFS endpoint files.

2. **Validate patient IDs:** Normalize patient IDs, check for duplicates and missing IDs, and verify that train, validation, and test patients do not overlap.

3. **Validate the RFS endpoint:** Check that patient IDs are unique, survival times are present and positive, and event labels contain only `0` or `1`.

4. **Verify clinical and genomic features:** Confirm that the feature tables match the corrected patient splits and that the genomic dataset contains exactly 200 selected genes.

5. **Use the authoritative RFS outcomes:** Remove the potentially outdated `survival_time` and `event` columns from the genomic feature table. Obtain both outcomes exclusively from the authoritative RFS endpoint.

6. **Match pathology data:** Use the pathology manifest to match each patient to the corresponding `.npy` feature array. Verify that all required pathology files exist.

7. **Construct the master dataset:** Merge the corrected patient splits, RFS outcomes, clinical features, genomic features, and pathology references using `patient_id`.

8. **Validate the final dataset:** Check patient uniqueness, outcome validity, split assignments, and pathology file availability. Report how many patients remain after matching all required data sources.

9. **Save the datasets:** Export the complete master dataset and separate train, validation, and test tables as CSV files.

## 4. Data Leakage Prevention and Important Corrections

- The corrected train, validation, and test assignments are preserved throughout master dataset construction.
- Clinical preprocessing and genomic feature selection are performed in their respective leakage-free pipelines, with learned preprocessing and feature selection restricted to the training data.
- The master dataset does not perform new feature selection or fit preprocessing transformations.
- The authoritative RFS endpoint is the only source of `survival_time` and `event`. This prevents outdated survival outcomes in the genomic CSV files from being used.
- Patient IDs are checked before merging, and one-to-one merge validation is used to prevent unintended duplication.
- Inner joins retain only patients who have matching records in all required data sources.
- Existing pathology arrays and source files are not modified.

**Important:** These checks help prevent data leakage and incorrect patient matching during master dataset construction. They do not, by themselves, guarantee that the entire modeling pipeline is free of leakage; that also depends on the preprocessing, feature selection, and model-training procedures.

## 5. Output Files

The final datasets are saved in:

`TCGA_BRCA/multimodal_rfs_dataset_leakage_free_v2/`

The notebook generates the following files:

- `master_rfs_all.csv` — Complete master dataset.
- `master_rfs_train.csv` — Training dataset.
- `master_rfs_validation.csv` — Validation dataset.
- `master_rfs_test.csv` — Test dataset.

## 6. Final Goal

The resulting dataset provides a consistent patient-level structure for multimodal Transformer-based RFS prediction. It combines clinical and genomic features with pathology feature references while preserving the corrected data splits and using consistent survival outcomes for model training and evaluation.

##Imports

In [1]:
# ============================================================
# IMPORT LIBRARIES
# gerekli kütüphaneleri yükledim
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from google.colab import drive

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

##Mount Google Drive

In [2]:
# ============================================================
# CONNECT GOOGLE DRIVE
# Google Drive'ı bağladım. Veri setlerini Google Drive üzerinde sakladım
# ============================================================

drive.mount("/content/drive")

Mounted at /content/drive


##Define Project Paths

In [15]:

from pathlib import Path
import pandas as pd
import numpy as np

# Main project directory
BASE_DIR = Path("/content/drive/MyDrive/TCGA_BRCA")

# Clinical preprocessing outputs
CLINICAL_DIR = BASE_DIR / "clinical_preprocessing_leakage_free"

# NEW: selected 200-gene features from corrected endpoint_v2 pipeline
GENOMIC_DIR = (
    BASE_DIR / "rfs_feature_selection_leakage_free_endpoint_v2"
)

# NEW: corrected patient splits and authoritative RFS outcomes
RFS_SPLIT_DIR = BASE_DIR / "dataset_split_rfs_endpoint_v2"

# Existing pathology features — do not modify
FEATURE_DIR = BASE_DIR / "Features"

# Authoritative RFS endpoint
FOLLOWUP_PATH = Path(
    "/content/drive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audited_v2_candidate.csv"
)

# New master dataset output folder
SAVE_DIR = BASE_DIR / "multimodal_rfs_dataset_leakage_free_v2"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

PATHOLOGY_FIXED_DIR = SAVE_DIR / "pathology_512"

required_paths = {
    "Clinical directory": CLINICAL_DIR,
    "Selected genomic directory": GENOMIC_DIR,
    "Corrected split directory": RFS_SPLIT_DIR,
    "Pathology feature directory": FEATURE_DIR,
    "Follow-up endpoint": FOLLOWUP_PATH,
}

print("PATH CHECK")
print("=" * 60)

for name, path in required_paths.items():
    print(f"{name}: {'EXISTS' if path.exists() else 'MISSING'}")
    print(f"  {path}")

assert all(path.exists() for path in required_paths.values()), (
    "A required source path is missing. Check the paths above."
)

print("\nMaster output directory:", SAVE_DIR)


PATH CHECK
Clinical directory: EXISTS
  /content/drive/MyDrive/TCGA_BRCA/clinical_preprocessing_leakage_free
Selected genomic directory: EXISTS
  /content/drive/MyDrive/TCGA_BRCA/rfs_feature_selection_leakage_free_endpoint_v2
Corrected split directory: EXISTS
  /content/drive/MyDrive/TCGA_BRCA/dataset_split_rfs_endpoint_v2
Pathology feature directory: EXISTS
  /content/drive/MyDrive/TCGA_BRCA/Features
Follow-up endpoint: EXISTS
  /content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint/followup_rfs_audited_v2_candidate.csv

Master output directory: /content/drive/MyDrive/TCGA_BRCA/multimodal_rfs_dataset_leakage_free_v2


##Load Clinical, Genomic and Follow-up

In [16]:

# ============================================================
# LOAD CORRECTED SPLITS AND FEATURE TABLES
# ============================================================

# 1. Load corrected patient split IDs
train_ids_df = pd.read_csv(
    RFS_SPLIT_DIR / "train_rfs_patient_ids.csv"
)

validation_ids_df = pd.read_csv(
    RFS_SPLIT_DIR / "validation_rfs_patient_ids.csv"
)

test_ids_df = pd.read_csv(
    RFS_SPLIT_DIR / "test_rfs_patient_ids.csv"
)

# Normalize IDs
for df in [train_ids_df, validation_ids_df, test_ids_df]:
    assert "patient_id" in df.columns
    df["patient_id"] = (
        df["patient_id"].astype(str).str.strip().str.upper()
    )
    assert df["patient_id"].is_unique
    assert df["patient_id"].notna().all()

train_ids = set(train_ids_df["patient_id"])
val_ids = set(validation_ids_df["patient_id"])
test_ids = set(test_ids_df["patient_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("Corrected split sizes:")
print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

assert (len(train_ids), len(val_ids), len(test_ids)) == (
    236, 49, 52
), "Unexpected corrected split sizes."

# 2. Load clinical features
clinical_train = pd.read_csv(
    CLINICAL_DIR / "clinical_train_leakage_free.csv"
)

clinical_validation = pd.read_csv(
    CLINICAL_DIR / "clinical_validation_leakage_free.csv"
)

clinical_test = pd.read_csv(
    CLINICAL_DIR / "clinical_test_leakage_free.csv"
)

# 3. Load the selected 200-gene genomic features
genomic_train = pd.read_csv(
    GENOMIC_DIR / "genomic_train_200_rfs_leakage_free.csv"
)

genomic_validation = pd.read_csv(
    GENOMIC_DIR / "genomic_validation_200_rfs_leakage_free.csv"
)

genomic_test = pd.read_csv(
    GENOMIC_DIR / "genomic_test_200_rfs_leakage_free.csv"
)

# 4. Normalize IDs in all feature tables
feature_tables = {
    "clinical_train": clinical_train,
    "clinical_validation": clinical_validation,
    "clinical_test": clinical_test,
    "genomic_train": genomic_train,
    "genomic_validation": genomic_validation,
    "genomic_test": genomic_test,
}

for name, df in feature_tables.items():
    assert "patient_id" in df.columns, (
        f"{name}: patient_id column is missing."
    )
    df["patient_id"] = (
        df["patient_id"].astype(str).str.strip().str.upper()
    )
    assert df["patient_id"].is_unique, (
        f"{name}: duplicate patient IDs."
    )

# 5. Filter clinical data to the corrected split membership
# This removes patients excluded from the corrected RFS cohort.
clinical_train = clinical_train[
    clinical_train["patient_id"].isin(train_ids)
].copy()

clinical_validation = clinical_validation[
    clinical_validation["patient_id"].isin(val_ids)
].copy()

clinical_test = clinical_test[
    clinical_test["patient_id"].isin(test_ids)
].copy()

# 6. Confirm every corrected split patient has clinical features
for name, df, expected_ids in [
    ("Clinical train", clinical_train, train_ids),
    ("Clinical validation", clinical_validation, val_ids),
    ("Clinical test", clinical_test, test_ids),
]:
    actual_ids = set(df["patient_id"])
    missing = expected_ids - actual_ids
    unexpected = actual_ids - expected_ids

    print(
        f"{name}: {len(df)} rows; "
        f"missing={len(missing)}; unexpected={len(unexpected)}"
    )

    if missing or unexpected:
        raise ValueError(
            f"{name} does not match the corrected split. "
            f"Missing examples: {list(sorted(missing))[:10]}"
        )

# 7. Confirm genomic data match the corrected splits
for name, df, expected_ids in [
    ("Genomic train", genomic_train, train_ids),
    ("Genomic validation", genomic_validation, val_ids),
    ("Genomic test", genomic_test, test_ids),
]:
    actual_ids = set(df["patient_id"])
    assert actual_ids == expected_ids, (
        f"{name} does not match the corrected split."
    )

# 8. Verify exactly 200 genes in each split
outcome_cols = {"patient_id", "survival_time", "event"}

gene_sets = {
    name: set(c for c in df.columns if c not in outcome_cols)
    for name, df in [
        ("train", genomic_train),
        ("validation", genomic_validation),
        ("test", genomic_test),
    ]
}

assert len(gene_sets["train"]) == 200
assert gene_sets["train"] == gene_sets["validation"]
assert gene_sets["train"] == gene_sets["test"]

print("\nPASS: Corrected splits and 200 genomic genes verified.")


Corrected split sizes:
Train: 236
Validation: 49
Test: 52
Clinical train: 236 rows; missing=0; unexpected=0
Clinical validation: 49 rows; missing=0; unexpected=0
Clinical test: 52 rows; missing=0; unexpected=0

PASS: Corrected splits and 200 genomic genes verified.


In [17]:

# ============================================================
# SECTION 4B: ESSENTIAL MASTER DATASET AUDIT
# Amaç: Veri kaynaklarını birleştirmeden önce kontrol etmek
# ============================================================

# 1. Check split IDs
def find_id_column(df, name):
    candidates = [
        c for c in df.columns
        if c.lower() in [
            "patient_id", "patient", "submitter_id",
            "case_id", "bcr_patient_barcode"
        ]
    ]

    if not candidates:
        raise ValueError(
            f"{name}: Patient ID column not found. "
            f"Available columns: {df.columns.tolist()}"
        )

    return candidates[0]


split_frames = {
    "train": train_ids_df,
    "validation": validation_ids_df,
    "test": test_ids_df,
}

split_ids = {}

print("=" * 70)
print("1. PATIENT SPLITS")
print("=" * 70)

for split_name, df in split_frames.items():
    id_col = find_id_column(df, split_name)
    ids = df[id_col].astype("string").str.strip()

    print(
        f"{split_name}: rows={len(df)}, "
        f"unique IDs={ids.nunique()}, "
        f"missing IDs={ids.isna().sum()}, "
        f"duplicates={ids.duplicated().sum()}"
    )

    split_ids[split_name] = set(ids.dropna())


print("\nCross-split overlap:")
for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    print(f"{a} vs {b}: {len(split_ids[a] & split_ids[b])}")


# 2. Clinical and genomic feature tables
print("\n" + "=" * 70)
print("2. CLINICAL AND GENOMIC TABLES")
print("=" * 70)

feature_tables = {
    "clinical_train": clinical_train,
    "clinical_validation": clinical_validation,
    "clinical_test": clinical_test,
    "genomic_train": genomic_train,
    "genomic_validation": genomic_validation,
    "genomic_test": genomic_test,
}

for name, df in feature_tables.items():
    id_col = find_id_column(df, name)
    ids = df[id_col].astype("string").str.strip()

    print(
        f"{name}: shape={df.shape}, ID column={id_col}, "
        f"unique IDs={ids.nunique()}, "
        f"duplicate IDs={ids.duplicated().sum()}, "
        f"missing IDs={ids.isna().sum()}"
    )


# 3. Compare clinical/genomic IDs with the assigned splits
print("\n" + "=" * 70)
print("3. SPLIT-TO-FEATURE ID MATCHING")
print("=" * 70)

for modality in ["clinical", "genomic"]:
    for split_name in ["train", "validation", "test"]:
        df = feature_tables[f"{modality}_{split_name}"]
        id_col = find_id_column(df, f"{modality}_{split_name}")
        ids = set(
            df[id_col].dropna().astype("string").str.strip()
        )

        expected = split_ids[split_name]

        print(
            f"{modality}_{split_name}: "
            f"missing from features={len(expected - ids)}, "
            f"unexpected IDs={len(ids - expected)}"
        )


# 4. Inspect RFS endpoint without changing it
print("\n" + "=" * 70)
print("4. RFS ENDPOINT")
print("=" * 70)

print("Shape:", followup_candidate.shape)
print("Columns:", followup_candidate.columns.tolist())

for col in followup_candidate.columns:
    if any(term in col.lower() for term in [
        "patient", "submitter", "barcode", "case_id",
        "survival_time", "event", "recurrence"
    ]):
        print(
            f"{col}: missing={followup_candidate[col].isna().sum()}, "
            f"unique={followup_candidate[col].nunique()}"
        )

if {"survival_time", "event"}.issubset(
    followup_candidate.columns
):
    print("\nEvent counts:")
    print(followup_candidate["event"].value_counts(dropna=False))

    print("\nSurvival-time summary:")
    print(followup_candidate["survival_time"].describe())


# 5. Inspect pathology files and metadata
print("\n" + "=" * 70)
print("5. PATHOLOGY INPUTS")
print("=" * 70)

print("Feature directory:", FEATURE_DIR)
print("Directory exists:", FEATURE_DIR.exists())

if FEATURE_DIR.exists():
    for path in sorted(FEATURE_DIR.iterdir()):
        if path.is_file():
            print(f"{path.name} ({path.stat().st_size:,} bytes)")
        elif path.is_dir():
            print(f"{path.name}/")

metadata_path = FEATURE_DIR / "features_metadata.csv"

if metadata_path.exists():
    pathology_metadata = pd.read_csv(metadata_path)

    print("\nPathology metadata shape:", pathology_metadata.shape)
    print("Metadata columns:", pathology_metadata.columns.tolist())
    display(pathology_metadata.head(5))

    for col in pathology_metadata.columns:
        if any(term in col.lower() for term in [
            "file_id", "submitter_id", "patient_id",
            "barcode", "case_id"
        ]):
            print(
                f"{col}: unique={pathology_metadata[col].nunique()}, "
                f"missing={pathology_metadata[col].isna().sum()}"
            )
else:
    print("features_metadata.csv was not found at the expected path.")

print("\nAudit finished. No source files were modified.")


1. PATIENT SPLITS
train: rows=236, unique IDs=236, missing IDs=0, duplicates=0
validation: rows=49, unique IDs=49, missing IDs=0, duplicates=0
test: rows=52, unique IDs=52, missing IDs=0, duplicates=0

Cross-split overlap:
train vs validation: 0
train vs test: 0
validation vs test: 0

2. CLINICAL AND GENOMIC TABLES
clinical_train: shape=(236, 21), ID column=patient_id, unique IDs=236, duplicate IDs=0, missing IDs=0
clinical_validation: shape=(49, 21), ID column=patient_id, unique IDs=49, duplicate IDs=0, missing IDs=0
clinical_test: shape=(52, 21), ID column=patient_id, unique IDs=52, duplicate IDs=0, missing IDs=0
genomic_train: shape=(236, 203), ID column=patient_id, unique IDs=236, duplicate IDs=0, missing IDs=0
genomic_validation: shape=(49, 203), ID column=patient_id, unique IDs=49, duplicate IDs=0, missing IDs=0
genomic_test: shape=(52, 203), ID column=patient_id, unique IDs=52, duplicate IDs=0, missing IDs=0

3. SPLIT-TO-FEATURE ID MATCHING
clinical_train: missing from features=

,patient_id,num_patches,feature_shape,num_patches_total,num_patches_accepted,num_patches_rejected,num_errors
0,a1b58cd0-559e-467d-a193-96602cd3306f,12697.0,"(12697, 512)",NaN,NaN,NaN,NaN
1,997dbd8f-9f16-4fc2-8627-d29ee0d74ebe,1989.0,"(1989, 512)",NaN,NaN,NaN,NaN
2,9132985f-ecd8-4a67-ba7b-63902801fba6,9047.0,"(9047, 512)",NaN,NaN,NaN,NaN
3,ce77af22-10b7-4480-9c93-015970268455,13609.0,"(13609, 512)",NaN,NaN,NaN,NaN
4,fcc84daa-5e54-4773-88a8-aa76727ffb6c,10695.0,"(10695, 512)",NaN,NaN,NaN,NaN


patient_id: unique=369, missing=0

Audit finished. No source files were modified.


In [18]:

# ============================================================
# SECTION 5: VALIDATE RFS ENDPOINT AND PATHOLOGY ID MATCHING
# ============================================================

# 1. Prepare the existing split IDs
train_ids = set(
    train_ids_df["patient_id"].dropna().astype(str).str.strip()
)
val_ids = set(
    validation_ids_df["patient_id"].dropna().astype(str).str.strip()
)
test_ids = set(
    test_ids_df["patient_id"].dropna().astype(str).str.strip()
)

split_ids = {
    "train": train_ids,
    "validation": val_ids,
    "test": test_ids,
}

# 2. Validate the RFS endpoint
print("=" * 70)
print("1. RFS ENDPOINT VALIDATION")
print("=" * 70)

required_endpoint_cols = {"patient_id", "survival_time", "event"}
missing_cols = required_endpoint_cols - set(followup_candidate.columns)

if missing_cols:
    raise ValueError(f"Missing endpoint columns: {missing_cols}")

endpoint = followup_candidate[
    ["patient_id", "survival_time", "event"]
].copy()

endpoint["patient_id"] = (
    endpoint["patient_id"].astype("string").str.strip()
)

endpoint["survival_time"] = pd.to_numeric(
    endpoint["survival_time"], errors="coerce"
)

endpoint["event"] = pd.to_numeric(
    endpoint["event"], errors="coerce"
)

print("Missing patient IDs:", endpoint["patient_id"].isna().sum())

# Remove rows with missing IDs before matching
endpoint = endpoint.dropna(subset=["patient_id"]).copy()

assert endpoint["patient_id"].is_unique, (
    "The RFS endpoint contains duplicate patient IDs."
)

assert endpoint["survival_time"].notna().all()
assert (endpoint["survival_time"] > 0).all()
assert endpoint["event"].notna().all()
assert endpoint["event"].isin([0, 1]).all()

endpoint_ids = set(endpoint["patient_id"])

print("\nEndpoint ID examples:")
print(endpoint["patient_id"].head(10).tolist())

# Check event and survival-time validity
print("\nInvalid survival times (<= 0):",
      (endpoint["survival_time"] <= 0).sum())

print("Invalid event labels (not 0 or 1):",
      (~endpoint["event"].isin([0, 1])).sum())

# 3. Check endpoint matching for each split
endpoint_ids = set(endpoint["patient_id"])

print("\n" + "=" * 70)
print("2. ENDPOINT MATCHING WITH EXISTING SPLITS")
print("=" * 70)

for split_name, ids in split_ids.items():
    matched = ids & endpoint_ids
    missing = ids - endpoint_ids

    print(f"\n{split_name.upper()}")
    print("Split patients:", len(ids))
    print("Matched endpoint patients:", len(matched))
    print("Missing endpoint patients:", len(missing))
    print("Missing ID examples:", list(sorted(missing))[:10])

# 4. Inspect pathology metadata and ID format
print("\n" + "=" * 70)
print("3. PATHOLOGY ID MATCHING")
print("=" * 70)

metadata_path = FEATURE_DIR / "features_metadata.csv"
pathology_metadata = pd.read_csv(metadata_path)

if "patient_id" not in pathology_metadata.columns:
    raise ValueError("Pathology metadata has no patient_id column.")

pathology_ids = set(
    pathology_metadata["patient_id"].dropna().astype(str).str.strip()
)

print("Pathology metadata rows:", len(pathology_metadata))
print("Unique pathology IDs:", len(pathology_ids))

print("\nPathology ID examples:")
print(list(sorted(pathology_ids))[:10])

for split_name, ids in split_ids.items():
    matched = ids & pathology_ids

    print(f"\n{split_name.upper()}")
    print("Split patients:", len(ids))
    print("Patients with pathology metadata:", len(matched))
    print("Patients without pathology metadata:", len(ids - pathology_ids))

# 5. Compare ID formats across all sources
print("\n" + "=" * 70)
print("4. ID FORMAT COMPARISON")
print("=" * 70)

for name, ids in [
    ("Train", train_ids),
    ("Endpoint", endpoint_ids),
    ("Pathology", pathology_ids),
]:
    print(f"\n{name} examples:")
    print(list(sorted(ids))[:5])

print("\nAudit complete. No source files were modified.")


1. RFS ENDPOINT VALIDATION
Missing patient IDs: 0

Endpoint ID examples:
['TCGA-3C-AAAU', 'TCGA-3C-AALI', 'TCGA-3C-AALJ', 'TCGA-3C-AALK', 'TCGA-4H-AAAK', 'TCGA-5L-AAT0', 'TCGA-5L-AAT1', 'TCGA-5T-A9QA', 'TCGA-A1-A0SB', 'TCGA-A1-A0SD']

Invalid survival times (<= 0): 0
Invalid event labels (not 0 or 1): 0

2. ENDPOINT MATCHING WITH EXISTING SPLITS

TRAIN
Split patients: 236
Matched endpoint patients: 236
Missing endpoint patients: 0
Missing ID examples: []

VALIDATION
Split patients: 49
Matched endpoint patients: 49
Missing endpoint patients: 0
Missing ID examples: []

TEST
Split patients: 52
Matched endpoint patients: 52
Missing endpoint patients: 0
Missing ID examples: []

3. PATHOLOGY ID MATCHING
Pathology metadata rows: 369
Unique pathology IDs: 369

Pathology ID examples:
['00ab866f-a0a3-44c0-ad14-c9dc833239a7', '01c79d06-a4a7-47a7-bb8b-b98ce6652c99', '02732da3-5144-4680-b6ee-cb8563125493', '0279f555-9738-41e0-b89f-ded25829adc0', '032fc1ed-c515-4b26-a229-f67c1eb6fdc3', '035f4d0d-78f

In [19]:

# ============================================================
# SECTION 6: FIND PATHOLOGY MAPPING FILES AND MISSING RFS IDS
# ============================================================

# 1. Save the split patients missing from the current endpoint
all_split_ids = train_ids | val_ids | test_ids
missing_rfs_ids = sorted(all_split_ids - endpoint_ids)

missing_rfs_df = pd.DataFrame({
    "patient_id": missing_rfs_ids
})

print("=" * 70)
print("1. PATIENTS MISSING FROM THE RFS ENDPOINT")
print("=" * 70)
display(missing_rfs_df)

# 2. Search likely project folders for mapping/manifest files
search_roots = [
    BASE_DIR,
    Path("/content/drive/MyDrive/BreastCanser-GenomicData"),
]

keywords = [
    "manifest", "metadata", "mapping", "slide",
    "pathology", "gdc", "file_id", "barcode"
]

candidate_files = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in [".csv", ".tsv", ".txt"]:
            continue

        name = path.name.lower()

        if any(word in name for word in keywords):
            candidate_files.append(path)

print("\n" + "=" * 70)
print("2. POSSIBLE MAPPING / METADATA FILES")
print("=" * 70)

for path in candidate_files[:100]:
    print(path)

print("\nCandidate files found:", len(candidate_files))

# 3. Inspect only headers and a few rows of likely mapping files
print("\n" + "=" * 70)
print("3. INSPECT POSSIBLE ID MAPPING TABLES")
print("=" * 70)

for path in candidate_files:
    try:
        if path.stat().st_size > 50 * 1024 * 1024:
            continue

        sep = "\t" if path.suffix.lower() == ".tsv" else ","
        sample = pd.read_csv(path, sep=sep, nrows=3, low_memory=False)

        cols = list(sample.columns)
        id_cols = [
            c for c in cols
            if any(term in c.lower() for term in [
                "uuid", "file_id", "file id", "patient",
                "submitter", "barcode", "case_id", "slide"
            ])
        ]

        # Show only files whose names or columns could help map IDs
        if id_cols or any(word in path.name.lower()
                          for word in ["manifest", "mapping", "metadata"]):
            print("\nFILE:", path)
            print("Columns:", cols)
            print("Possible ID columns:", id_cols)
            display(sample.head(3))

    except Exception as exc:
        print("Could not inspect:", path.name, "|", str(exc)[:150])

print("\nInspection complete. No source files were modified.")


1. PATIENTS MISSING FROM THE RFS ENDPOINT


,patient_id



2. POSSIBLE MAPPING / METADATA FILES
/content/drive/MyDrive/TCGA_BRCA/Features/features_metadata.csv
/content/drive/MyDrive/TCGA_BRCA/explainability_results_v1/top_pathology_attention_patches.csv
/content/drive/MyDrive/TCGA_BRCA/explainability_results_v1/pathology_top_patch_attention.csv
/content/drive/MyDrive/TCGA_BRCA/multimodal_rfs_dataset_leakage_free/pathology_512/pathology_512_manifest.csv

Candidate files found: 4

3. INSPECT POSSIBLE ID MAPPING TABLES

FILE: /content/drive/MyDrive/TCGA_BRCA/Features/features_metadata.csv
Columns: ['patient_id', 'num_patches', 'feature_shape', 'num_patches_total', 'num_patches_accepted', 'num_patches_rejected', 'num_errors']
Possible ID columns: ['patient_id']


,patient_id,num_patches,feature_shape,num_patches_total,num_patches_accepted,num_patches_rejected,num_errors
0,a1b58cd0-559e-467d-a193-96602cd3306f,12697.0,"(12697, 512)",NaN,NaN,NaN,NaN
1,997dbd8f-9f16-4fc2-8627-d29ee0d74ebe,1989.0,"(1989, 512)",NaN,NaN,NaN,NaN
2,9132985f-ecd8-4a67-ba7b-63902801fba6,9047.0,"(9047, 512)",NaN,NaN,NaN,NaN



FILE: /content/drive/MyDrive/TCGA_BRCA/multimodal_rfs_dataset_leakage_free/pathology_512/pathology_512_manifest.csv
Columns: ['patient_id', 'file_uuid', 'original_num_patches', 'selected_num_patches', 'padded_patches', 'feature_dim']
Possible ID columns: ['patient_id', 'file_uuid']


,patient_id,file_uuid,original_num_patches,selected_num_patches,padded_patches,feature_dim
0,TCGA-5L-AAT1,4eec69ca-381b-4c17-b3e9-49492d71560e,1777,512,0,512
1,TCGA-A1-A0SB,cea82b7d-135a-49d5-b4f6-3fb0215f7188,4512,512,0,512
2,TCGA-A1-A0SF,f273a858-1cfa-4983-ace4-b1e876d64473,6094,512,0,512



Inspection complete. No source files were modified.


In [20]:
from pathlib import Path
import pandas as pd


manifest_candidates = [
    BASE_DIR / "multimodal_rfs_dataset_leakage_free" / "pathology_512" / "pathology_512_manifest.csv",
    BASE_DIR / "multimodal_rfs_dataset_leakage_free_v2" / "pathology_512" / "pathology_512_manifest.csv",
]

manifest_path = next(
    (p for p in manifest_candidates if p.exists()),
    None
)

if manifest_path is None:
    raise FileNotFoundError(
        "Manifest not found in either expected location. "
        "Check the folder containing pathology_512_manifest.csv."
    )

manifest = pd.read_csv(manifest_path)
manifest["patient_id"] = manifest["patient_id"].astype(str).str.strip()
manifest["file_uuid"] = manifest["file_uuid"].astype(str).str.strip()

print("Manifest loaded:", manifest_path)
print("Manifest rows:", len(manifest))
print("Unique patients:", manifest["patient_id"].nunique())
print("Duplicate patient IDs:", manifest["patient_id"].duplicated().sum())
print("Duplicate UUIDs:", manifest["file_uuid"].duplicated().sum())
print(manifest.head())

Manifest loaded: /content/drive/MyDrive/TCGA_BRCA/multimodal_rfs_dataset_leakage_free/pathology_512/pathology_512_manifest.csv
Manifest rows: 357
Unique patients: 357
Duplicate patient IDs: 0
Duplicate UUIDs: 0
     patient_id                             file_uuid  original_num_patches  \
0  TCGA-5L-AAT1  4eec69ca-381b-4c17-b3e9-49492d71560e                  1777   
1  TCGA-A1-A0SB  cea82b7d-135a-49d5-b4f6-3fb0215f7188                  4512   
2  TCGA-A1-A0SF  f273a858-1cfa-4983-ace4-b1e876d64473                  6094   
3  TCGA-A1-A0SI  a951559a-e6c0-419b-b09f-43e6cb6ee3ea                  6822   
4  TCGA-A1-A0SJ  9a3a3830-9674-4f9b-985f-e50062e78807                  4801   

   selected_num_patches  padded_patches  feature_dim  
0                   512               0          512  
1                   512               0          512  
2                   512               0          512  
3                   512               0          512  
4                   512               0

In [21]:
from pathlib import Path

search_dirs = [
    FEATURE_DIR,
    BASE_DIR / "multimodal_rfs_dataset_leakage_free" / "pathology_512",
    BASE_DIR / "multimodal_rfs_dataset_leakage_free_v2" / "pathology_512",
]

npy_paths = []
for folder in search_dirs:
    if folder.exists():
        npy_paths.extend(folder.rglob("*.npy"))

available_stems = {p.stem for p in npy_paths}

manifest["npy_file_exists"] = (
    manifest["file_uuid"].isin(available_stems)
    | manifest["patient_id"].isin(available_stems)
)

pathology_ids = set(
    manifest.loc[manifest["npy_file_exists"], "patient_id"]
)

endpoint_ids = set(followup_candidate["patient_id"].astype(str).str.strip())
split_ids = train_ids | val_ids | test_ids
eligible_ids = split_ids & endpoint_ids & pathology_ids

print("Pathology .npy files found:", len(npy_paths))
print("Manifest entries with matching arrays:", int(manifest["npy_file_exists"].sum()))
print("Manifest entries without matching arrays:", int((~manifest["npy_file_exists"]).sum()))

print("\nELIGIBLE PATIENTS")
print("All splits:", len(eligible_ids))
print("Train:", len(eligible_ids & train_ids))
print("Validation:", len(eligible_ids & val_ids))
print("Test:", len(eligible_ids & test_ids))

print("\nMissing RFS endpoints:", len(split_ids - endpoint_ids))
print("Audit complete.")

Pathology .npy files found: 1086
Manifest entries with matching arrays: 357
Manifest entries without matching arrays: 0

ELIGIBLE PATIENTS
All splits: 337
Train: 236
Validation: 49
Test: 52

Missing RFS endpoints: 0
Audit complete.


##CREATE master dataset:

In [23]:
# Diagnose mismatches between genomic CSV outcomes and the RFS endpoint

comparison = genomic_all[
    ["patient_id", "survival_time", "event"]
].merge(
    endpoint[["patient_id", "survival_time", "event"]],
    on="patient_id",
    how="left",
    suffixes=("_genomic", "_endpoint"),
    validate="one_to_one",
    indicator=True
)

print("Genomic patients:", len(comparison))
print("Missing endpoint matches:", (comparison["_merge"] != "both").sum())

comparison["time_difference"] = (
    pd.to_numeric(comparison["survival_time_genomic"], errors="coerce")
    - pd.to_numeric(comparison["survival_time_endpoint"], errors="coerce")
).abs()

time_mismatch = comparison[
    ~np.isclose(
        comparison["survival_time_genomic"],
        comparison["survival_time_endpoint"],
        equal_nan=False
    )
]

event_mismatch = comparison[
    comparison["event_genomic"] != comparison["event_endpoint"]
]

print("\nSurvival-time mismatches:", len(time_mismatch))
print("Event-label mismatches:", len(event_mismatch))

print("\nFirst 20 survival-time mismatches:")
display(time_mismatch[
    [
        "patient_id",
        "survival_time_genomic",
        "survival_time_endpoint",
        "time_difference",
        "event_genomic",
        "event_endpoint"
    ]
].head(20))

print("\nFirst 20 event-label mismatches:")
display(event_mismatch[
    [
        "patient_id",
        "event_genomic",
        "event_endpoint"
    ]
].head(20))

Genomic patients: 337
Missing endpoint matches: 0

Survival-time mismatches: 23
Event-label mismatches: 0

First 20 survival-time mismatches:


,patient_id,survival_time_genomic,survival_time_endpoint,time_difference,event_genomic,event_endpoint
240,TCGA-A2-A25E,3204.0,2989.0,215.0,1,1
241,TCGA-A2-A3XS,1032.0,682.0,350.0,1,1
247,TCGA-AC-A2FE,2636.0,766.0,1870.0,1,1
248,TCGA-AC-A2QJ,446.0,237.0,209.0,1,1
256,TCGA-AO-A0J9,1613.0,1066.0,547.0,1,1
259,TCGA-AR-A0TZ,3262.0,1313.0,1949.0,1,1
261,TCGA-B6-A0RH,6456.0,2165.0,4291.0,1,1
263,TCGA-BH-A0BQ,2255.0,1709.0,546.0,1,1
272,TCGA-E2-A1IE,2362.0,1810.0,552.0,1,1
276,TCGA-E9-A1NF,1072.0,90.0,982.0,1,1



First 20 event-label mismatches:


,patient_id,event_genomic,event_endpoint


In [24]:

# ============================================================
# BUILD LEAKAGE-FREE MASTER DATASET
# Clinical + Genomic + RFS + Pathology references
# Uses the authoritative RFS endpoint for survival outcomes.
# Does not modify source files or pathology arrays.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 0. Output directory
# ------------------------------------------------------------

OUTPUT_DIR = SAVE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Prepare the authoritative RFS endpoint
# ------------------------------------------------------------

endpoint = followup_candidate[
    ["patient_id", "survival_time", "event"]
].copy()

endpoint["patient_id"] = (
    endpoint["patient_id"].astype("string").str.strip()
)
endpoint["survival_time"] = pd.to_numeric(
    endpoint["survival_time"], errors="coerce"
)
endpoint["event"] = pd.to_numeric(
    endpoint["event"], errors="coerce"
)

assert endpoint["patient_id"].notna().all(), (
    "Missing patient IDs in the RFS endpoint."
)
assert endpoint["patient_id"].is_unique, (
    "Duplicate endpoint patient IDs."
)
assert endpoint["survival_time"].notna().all(), (
    "Missing survival times in the RFS endpoint."
)
assert endpoint["event"].notna().all(), (
    "Missing event labels in the RFS endpoint."
)
assert (endpoint["survival_time"] > 0).all(), (
    "Non-positive survival times in the RFS endpoint."
)
assert endpoint["event"].isin([0, 1]).all(), (
    "Event labels must be 0 or 1."
)

print("PASS: authoritative RFS endpoint validated.")
print("Endpoint patients:", len(endpoint))
print("Endpoint event counts:")
print(endpoint["event"].value_counts().sort_index().to_dict())

# ------------------------------------------------------------
# 2. Combine clinical and genomic feature tables
# ------------------------------------------------------------

clinical_all = pd.concat(
    [clinical_train, clinical_validation, clinical_test],
    ignore_index=True
).copy()

genomic_all = pd.concat(
    [genomic_train, genomic_validation, genomic_test],
    ignore_index=True
).copy()

# Normalize patient IDs and verify uniqueness.
for name, df in [
    ("Clinical", clinical_all),
    ("Genomic", genomic_all)
]:
    assert "patient_id" in df.columns, (
        f"{name} table has no patient_id column."
    )

    df["patient_id"] = (
        df["patient_id"].astype("string").str.strip()
    )

    assert df["patient_id"].notna().all(), (
        f"Missing patient IDs in {name} table."
    )
    assert df["patient_id"].is_unique, (
        f"Duplicate patient IDs in {name} table."
    )

print("\nClinical patients:", len(clinical_all))
print("Genomic patients:", len(genomic_all))

# ------------------------------------------------------------
# 3. Verify genomic patient IDs against the endpoint
#
# IMPORTANT:
# The genomic CSVs contain 23 stale/different survival times.
# Therefore, do not use their survival_time/event columns.
# The master dataset will obtain these outcomes ONLY from
# the authoritative endpoint prepared in Section 1.
# ------------------------------------------------------------

genomic_id_check = genomic_all[
    ["patient_id"]
].merge(
    endpoint[["patient_id"]],
    on="patient_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

missing_endpoint_ids = genomic_id_check.loc[
    genomic_id_check["_merge"] != "both",
    "patient_id"
].tolist()

assert len(missing_endpoint_ids) == 0, (
    "Some genomic patients are missing from the authoritative "
    f"RFS endpoint. Examples: {missing_endpoint_ids[:10]}"
)

print(
    "PASS: all genomic patient IDs match the authoritative endpoint."
)

# Remove potentially outdated outcomes from the genomic feature table.
# They will be supplied by endpoint during the master merge.
outcome_columns = [
    col for col in ["survival_time", "event"]
    if col in genomic_all.columns
]

genomic_all = genomic_all.drop(
    columns=outcome_columns
)

print("Removed genomic outcome columns:", outcome_columns)
print("Genomic feature columns retained:", len(genomic_all.columns) - 1)

# Check selected genomic feature count when the expected setup is 200 genes.
genomic_feature_columns = [
    col for col in genomic_all.columns
    if col != "patient_id"
]

assert len(genomic_feature_columns) == 200, (
    "Expected 200 selected genomic features, but found "
    f"{len(genomic_feature_columns)}."
)

assert genomic_all[genomic_feature_columns].notna().all().all(), (
    "Missing values detected in genomic features."
)

# ------------------------------------------------------------
# 4. Prepare pathology references
# ------------------------------------------------------------

pathology = manifest[
    ["patient_id", "file_uuid"]
].copy()

pathology["patient_id"] = (
    pathology["patient_id"].astype("string").str.strip()
)
pathology["file_uuid"] = (
    pathology["file_uuid"].astype("string").str.strip()
)

assert pathology["patient_id"].notna().all(), (
    "Missing patient IDs in pathology manifest."
)
assert pathology["file_uuid"].notna().all(), (
    "Missing file UUIDs in pathology manifest."
)
assert pathology["patient_id"].is_unique, (
    "Duplicate pathology patient IDs."
)
assert pathology["file_uuid"].is_unique, (
    "Duplicate pathology file UUIDs."
)

# Find arrays by UUID in the previously audited folders.
array_lookup = {}

pathology_folders = [
    FEATURE_DIR,
    BASE_DIR / "multimodal_rfs_dataset_leakage_free" / "pathology_512",
    BASE_DIR / "multimodal_rfs_dataset_leakage_free_v2" / "pathology_512",
]

for folder in pathology_folders:
    if folder.exists():
        for path in folder.rglob("*.npy"):
            # A UUID should identify one array. Do not silently
            # choose between duplicate UUIDs in different folders.
            if path.stem in array_lookup:
                existing_path = Path(array_lookup[path.stem])
                if existing_path.resolve() != path.resolve():
                    raise ValueError(
                        "The same pathology UUID was found at multiple "
                        f"paths:\n{existing_path}\n{path}"
                    )
            else:
                array_lookup[path.stem] = str(path)

print("\nPathology .npy files indexed:", len(array_lookup))
print("Pathology manifest patients:", len(pathology))

pathology["pathology_path"] = pathology["file_uuid"].map(
    array_lookup
)

missing_pathology_arrays = pathology.loc[
    pathology["pathology_path"].isna(),
    ["patient_id", "file_uuid"]
]

assert missing_pathology_arrays.empty, (
    "Some pathology arrays could not be located by UUID. "
    f"Examples:\n{missing_pathology_arrays.head(10)}"
)

print("PASS: all pathology manifest entries have matching arrays.")

# ------------------------------------------------------------
# 5. Prepare corrected splits and verify no overlap
# ------------------------------------------------------------

split_table = pd.concat(
    [
        pd.DataFrame({
            "patient_id": list(train_ids),
            "split": "train"
        }),
        pd.DataFrame({
            "patient_id": list(val_ids),
            "split": "validation"
        }),
        pd.DataFrame({
            "patient_id": list(test_ids),
            "split": "test"
        }),
    ],
    ignore_index=True
)

split_table["patient_id"] = (
    split_table["patient_id"].astype("string").str.strip()
)

assert split_table["patient_id"].notna().all(), (
    "Missing patient IDs in split table."
)
assert split_table["patient_id"].is_unique, (
    "A patient appears in more than one split."
)

print("\nCorrected split sizes:")
print(split_table["split"].value_counts().reindex(
    ["train", "validation", "test"],
    fill_value=0
))

# ------------------------------------------------------------
# 6. Build master dataset
#
# Inner joins retain patients present in ALL required sources:
# corrected split, endpoint, clinical, genomic and pathology.
# ------------------------------------------------------------

master = (
    split_table
    .merge(
        endpoint,
        on="patient_id",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        clinical_all,
        on="patient_id",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_clinical")
    )
    .merge(
        genomic_all,
        on="patient_id",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_genomic")
    )
    .merge(
        pathology,
        on="patient_id",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_pathology")
    )
)

# ------------------------------------------------------------
# 7. Verify master dataset integrity
# ------------------------------------------------------------

assert not master.empty, (
    "Master dataset is empty. Check patient-ID overlap across modalities."
)
assert master["patient_id"].is_unique, (
    "Duplicate patient IDs in master dataset."
)
assert master["survival_time"].notna().all(), (
    "Missing authoritative survival times in master dataset."
)
assert master["event"].notna().all(), (
    "Missing authoritative event labels in master dataset."
)
assert (master["survival_time"] > 0).all(), (
    "Invalid survival times in master dataset."
)
assert master["event"].isin([0, 1]).all(), (
    "Invalid event labels in master dataset."
)

# Every retained patient must belong to exactly one corrected split.
assert master["split"].isin(
    ["train", "validation", "test"]
).all()

# Report patients excluded because not all modalities were available.
print("\nMaster dataset shape:", master.shape)

print("\nPatients per split:")
print(
    master["split"].value_counts().reindex(
        ["train", "validation", "test"],
        fill_value=0
    )
)

print("\nEvent counts by split:")
print(pd.crosstab(
    master["split"],
    master["event"]
).reindex(
    index=["train", "validation", "test"],
    columns=[0, 1],
    fill_value=0
))

print("\nMaster patients compared with corrected split:")
print("Corrected split patients:", len(split_table))
print("Patients retained in master:", len(master))
print("Patients not retained:", len(split_table) - len(master))

# ------------------------------------------------------------
# 8. Save full master table and split-specific tables
# ------------------------------------------------------------

master.to_csv(
    OUTPUT_DIR / "master_rfs_all.csv",
    index=False
)

for split_name in ["train", "validation", "test"]:
    part = master.loc[
        master["split"] == split_name
    ].copy()

    part.to_csv(
        OUTPUT_DIR / f"master_rfs_{split_name}.csv",
        index=False
    )

print("\nSaved master tables to:", OUTPUT_DIR)

for filename in [
    "master_rfs_all.csv",
    "master_rfs_train.csv",
    "master_rfs_validation.csv",
    "master_rfs_test.csv"
]:
    filepath = OUTPUT_DIR / filename
    if filepath.exists():
        row_count = len(pd.read_csv(filepath, usecols=["patient_id"]))
        print(f"- {filename}: {row_count} patients")

print("\nMASTER DATASET BUILD COMPLETED.")


PASS: authoritative RFS endpoint validated.
Endpoint patients: 1062
Endpoint event counts:
{0: 979, 1: 83}

Clinical patients: 337
Genomic patients: 337
PASS: all genomic patient IDs match the authoritative endpoint.
Removed genomic outcome columns: ['survival_time', 'event']
Genomic feature columns retained: 200

Pathology .npy files indexed: 1086
Pathology manifest patients: 357
PASS: all pathology manifest entries have matching arrays.

Corrected split sizes:
split
train         236
validation     49
test           52
Name: count, dtype: int64

Master dataset shape: (337, 226)

Patients per split:
split
train         236
validation     49
test           52
Name: count, dtype: int64

Event counts by split:
event         0   1
split              
train       182  54
validation   37  12
test         40  12

Master patients compared with corrected split:
Corrected split patients: 337
Patients retained in master: 337
Patients not retained: 0

Saved master tables to: /content/drive/MyDriv

In [11]:

# ============================================================
# DIAGNOSE GENOMIC PATIENTS MISSING FROM THE RFS ENDPOINT
# This cell only checks IDs; it does not modify source files.
# ============================================================

# Normalize IDs for comparison
genomic_ids = genomic_all["patient_id"].astype("string").str.strip()
endpoint_ids_clean = endpoint["patient_id"].astype("string").str.strip()

genomic_check = genomic_all[
    ["patient_id", "survival_time", "event"]
].copy()
genomic_check["patient_id"] = genomic_ids

endpoint_check = endpoint[
    ["patient_id", "survival_time", "event"]
].copy()
endpoint_check["patient_id"] = endpoint_ids_clean

comparison = genomic_check.merge(
    endpoint_check,
    on="patient_id",
    how="left",
    suffixes=("_genomic", "_endpoint"),
    validate="one_to_one",
    indicator=True
)

missing = comparison.loc[
    comparison["_merge"] == "left_only",
    ["patient_id", "survival_time_genomic", "event_genomic"]
].copy()

print("=" * 70)
print("EXACT PATIENT-ID MATCH AUDIT")
print("=" * 70)
print("Genomic patients:", len(genomic_check))
print("Endpoint patients:", len(endpoint_check))
print("Exact ID matches:", int((comparison["_merge"] == "both").sum()))
print("Genomic patients missing from endpoint:", len(missing))

print("\nFirst 20 unmatched genomic patients:")
display(missing.head(20))

# Check whether unmatched IDs might differ only by TCGA sample/barcode format.
# For TCGA IDs, the first three hyphen-separated parts identify the patient.
def tcga_patient_key(series):
    return (
        series.astype("string")
        .str.strip()
        .str.split("-")
        .str[:3]
        .str.join("-")
    )

genomic_keys = set(tcga_patient_key(genomic_check["patient_id"]).dropna())
endpoint_keys = set(tcga_patient_key(endpoint_check["patient_id"]).dropna())

missing["tcga_patient_key"] = tcga_patient_key(missing["patient_id"])
missing["key_found_in_endpoint"] = missing["tcga_patient_key"].isin(endpoint_keys)

print("\nPotential barcode-format differences:")
print(
    "Unmatched IDs whose first three TCGA barcode parts exist in endpoint:",
    int(missing["key_found_in_endpoint"].sum())
)

print("\nExamples where the patient key exists in the endpoint:")
display(
    missing.loc[missing["key_found_in_endpoint"]].head(20)
)

print("\nExamples where even the patient key is absent:")
display(
    missing.loc[~missing["key_found_in_endpoint"]].head(20)
)

print("\nDiagnostic complete. No source files were modified.")


EXACT PATIENT-ID MATCH AUDIT
Genomic patients: 357
Endpoint patients: 1062
Exact ID matches: 337
Genomic patients missing from endpoint: 20

First 20 unmatched genomic patients:


,patient_id,survival_time_genomic,event_genomic
0,TCGA-A2-A0SX,1534.0,1
6,TCGA-A8-A08O,943.0,1
13,TCGA-AR-A24O,3607.0,1
15,TCGA-A2-A04V,1920.0,1
33,TCGA-AR-A0TW,3009.0,1
48,TCGA-A2-A0SW,1365.0,1
77,TCGA-D8-A1XC,377.0,1
83,TCGA-A8-A091,1004.0,1
107,TCGA-A8-A08T,3409.0,1
137,TCGA-A8-A094,1.0,0



Potential barcode-format differences:
Unmatched IDs whose first three TCGA barcode parts exist in endpoint: 0

Examples where the patient key exists in the endpoint:


,patient_id,survival_time_genomic,event_genomic,tcga_patient_key,key_found_in_endpoint



Examples where even the patient key is absent:


,patient_id,survival_time_genomic,event_genomic,tcga_patient_key,key_found_in_endpoint
0,TCGA-A2-A0SX,1534.0,1,TCGA-A2-A0SX,False
6,TCGA-A8-A08O,943.0,1,TCGA-A8-A08O,False
13,TCGA-AR-A24O,3607.0,1,TCGA-AR-A24O,False
15,TCGA-A2-A04V,1920.0,1,TCGA-A2-A04V,False
33,TCGA-AR-A0TW,3009.0,1,TCGA-AR-A0TW,False
48,TCGA-A2-A0SW,1365.0,1,TCGA-A2-A0SW,False
77,TCGA-D8-A1XC,377.0,1,TCGA-D8-A1XC,False
83,TCGA-A8-A091,1004.0,1,TCGA-A8-A091,False
107,TCGA-A8-A08T,3409.0,1,TCGA-A8-A08T,False
137,TCGA-A8-A094,1.0,0,TCGA-A8-A094,False



Diagnostic complete. No source files were modified.


In [12]:
from pathlib import Path
import pandas as pd

RFS_DIR = Path(
    "/content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint"
)

print("RFS endpoint files:")
for p in sorted(RFS_DIR.glob("*")):
    if p.is_file():
        print(p.name)

RFS endpoint files:
followup_rfs_audit_all_patients.csv
followup_rfs_audited_v2_candidate.csv
followup_rfs_final_v1.csv


In [13]:

# READ-ONLY CHECK: inspect existing master-dataset inputs
import pandas as pd

clinical_all_check = pd.concat(
    [clinical_train, clinical_validation, clinical_test],
    ignore_index=True
)
genomic_all_check = pd.concat(
    [genomic_train, genomic_validation, genomic_test],
    ignore_index=True
)

def clean_ids(df):
    return set(
        df["patient_id"]
        .astype("string")
        .str.strip()
        .str.upper()
        .dropna()
    )

endpoint_check_ids = clean_ids(followup_candidate)
clinical_ids = clean_ids(clinical_all_check)
genomic_ids = clean_ids(genomic_all_check)

print("Final endpoint patients:", len(endpoint_check_ids))
print("Clinical patients:", len(clinical_ids))
print("Genomic patients:", len(genomic_ids))

print("\nGenomic patients absent from final endpoint:",
      len(genomic_ids - endpoint_check_ids))
print("Clinical patients absent from final endpoint:",
      len(clinical_ids - endpoint_check_ids))

print("\nFinal endpoint patients missing from genomic:",
      len(endpoint_check_ids - genomic_ids))
print("Final endpoint patients missing from clinical:",
      len(endpoint_check_ids - clinical_ids))

print("\nGenomic patients absent from endpoint (first 30):")
print(sorted(genomic_ids - endpoint_check_ids)[:30])

print("\nClinical patients absent from endpoint (first 30):")
print(sorted(clinical_ids - endpoint_check_ids)[:30])

print("\nNo files were modified.")


Final endpoint patients: 1062
Clinical patients: 357
Genomic patients: 357

Genomic patients absent from final endpoint: 20
Clinical patients absent from final endpoint: 20

Final endpoint patients missing from genomic: 725
Final endpoint patients missing from clinical: 725

Genomic patients absent from endpoint (first 30):
['TCGA-A2-A04V', 'TCGA-A2-A0CM', 'TCGA-A2-A0SW', 'TCGA-A2-A0SX', 'TCGA-A8-A06N', 'TCGA-A8-A06Y', 'TCGA-A8-A08O', 'TCGA-A8-A08T', 'TCGA-A8-A08X', 'TCGA-A8-A091', 'TCGA-A8-A094', 'TCGA-A8-A096', 'TCGA-AC-A23H', 'TCGA-AC-A3W6', 'TCGA-AR-A0TW', 'TCGA-AR-A24O', 'TCGA-C8-A12Q', 'TCGA-C8-A12T', 'TCGA-C8-A26Y', 'TCGA-D8-A1XC']

Clinical patients absent from endpoint (first 30):
['TCGA-A2-A04V', 'TCGA-A2-A0CM', 'TCGA-A2-A0SW', 'TCGA-A2-A0SX', 'TCGA-A8-A06N', 'TCGA-A8-A06Y', 'TCGA-A8-A08O', 'TCGA-A8-A08T', 'TCGA-A8-A08X', 'TCGA-A8-A091', 'TCGA-A8-A094', 'TCGA-A8-A096', 'TCGA-AC-A23H', 'TCGA-AC-A3W6', 'TCGA-AR-A0TW', 'TCGA-AR-A24O', 'TCGA-C8-A12Q', 'TCGA-C8-A12T', 'TCGA-C8-A26

In [14]:

import os
import pandas as pd

# Use the final endpoint you already checked
endpoint = followup_candidate[
    ["patient_id", "survival_time", "event"]
].copy()

endpoint["patient_id"] = (
    endpoint["patient_id"].astype(str).str.strip().str.upper()
)

assert endpoint["patient_id"].is_unique, "Duplicate IDs in endpoint"
assert endpoint["survival_time"].notna().all(), "Missing survival times"
assert endpoint["event"].isin([0, 1]).all(), "Invalid event values"

# Keep the existing split assignments
split_ids = {
    "train": train_ids,
    "validation": val_ids,
    "test": test_ids,
}

OUTPUT_SPLIT_DIR = (
    "/content/drive/MyDrive/TCGA_BRCA/"
    "dataset_split_rfs_endpoint_v2"
)
os.makedirs(OUTPUT_SPLIT_DIR, exist_ok=True)

endpoint_ids = set(endpoint["patient_id"])
corrected = {}

for split_name, ids in split_ids.items():
    ids = {
        str(patient_id).strip().upper()
        for patient_id in ids
    }

    # Keep only patients present in the final endpoint
    eligible_ids = ids.intersection(endpoint_ids)

    split_df = endpoint[
        endpoint["patient_id"].isin(eligible_ids)
    ].copy()

    split_df = split_df.sort_values("patient_id").reset_index(drop=True)
    corrected[split_name] = split_df

    # Save corrected outcomes
    split_df.to_csv(
        os.path.join(
            OUTPUT_SPLIT_DIR,
            f"{split_name}_rfs.csv"
        ),
        index=False
    )

    # Save patient IDs separately
    split_df[["patient_id"]].to_csv(
        os.path.join(
            OUTPUT_SPLIT_DIR,
            f"{split_name}_rfs_patient_ids.csv"
        ),
        index=False
    )

    print(
        f"{split_name}: {len(ids)} original IDs -> "
        f"{len(split_df)} eligible patients"
    )

# Check the corrected splits
all_corrected_ids = [
    patient_id
    for df in corrected.values()
    for patient_id in df["patient_id"]
]

assert len(all_corrected_ids) == len(set(all_corrected_ids)), (
    "A patient appears in more than one split."
)

print("\nCorrected total:", len(all_corrected_ids))
print("Saved to:", OUTPUT_SPLIT_DIR)
print("\nEvent counts:")
for split_name, df in corrected.items():
    print(split_name, df["event"].value_counts().to_dict())


train: 249 original IDs -> 236 eligible patients
validation: 54 original IDs -> 49 eligible patients
test: 54 original IDs -> 52 eligible patients

Corrected total: 337
Saved to: /content/drive/MyDrive/TCGA_BRCA/dataset_split_rfs_endpoint_v2

Event counts:
train {0: 182, 1: 54}
validation {0: 37, 1: 12}
test {0: 40, 1: 12}


##Verify master dataset:

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path

# MASTER_DIR = BASE_DIR / "master_dataset_rfs_leakage_free"
# master_check = pd.read_csv(MASTER_DIR / "master_rfs_all.csv")

MASTER_DIR = BASE_DIR / "multimodal_rfs_dataset_leakage_free_v2"
master_check = pd.read_csv(MASTER_DIR / "master_rfs_all.csv")

# 1. Check patient IDs and split sizes
print("Rows:", len(master_check))
print("Unique patients:", master_check["patient_id"].nunique())
print("\nSplit counts:")
print(master_check["split"].value_counts())

assert master_check["patient_id"].is_unique
assert set(master_check["split"]) == {"train", "validation", "test"}

# 2. Check RFS outcomes
print("\nMissing survival times:", master_check["survival_time"].isna().sum())
print("Missing event labels:", master_check["event"].isna().sum())
print("Invalid survival times:",
      (pd.to_numeric(master_check["survival_time"], errors="coerce") <= 0).sum())
print("\nEvent counts:")
print(pd.crosstab(master_check["split"], master_check["event"]))

assert master_check["survival_time"].notna().all()
assert master_check["event"].notna().all()
assert (master_check["survival_time"] > 0).all()
assert master_check["event"].isin([0, 1]).all()

# 3. Check pathology file references
pathology_paths = master_check["pathology_path"].dropna().map(Path)
missing_paths = [str(p) for p in pathology_paths if not p.is_file()]

print("\nMissing pathology files:", len(missing_paths))
assert len(pathology_paths) == len(master_check)
assert len(missing_paths) == 0

# 4. Inspect a few pathology arrays
for path in pathology_paths.head(3):
    arr = np.load(path, mmap_mode="r")
    print(path.name, "| shape:", arr.shape, "| dtype:", arr.dtype)

print("\nVERIFICATION PASSED")

Rows: 337
Unique patients: 337

Split counts:
split
train         236
test           52
validation     49
Name: count, dtype: int64

Missing survival times: 0
Missing event labels: 0
Invalid survival times: 0

Event counts:
event         0   1
split              
test         40  12
train       182  54
validation   37  12

Missing pathology files: 0
3687e120-6235-4d3c-8176-543a7c6a0014.npy | shape: (1316, 512) | dtype: float32
81990d74-6013-4a15-a18a-62dac734d8e5.npy | shape: (2329, 512) | dtype: float32
ebef3a2a-227f-40f7-9a6d-5138fa43c872.npy | shape: (4187, 512) | dtype: float32

VERIFICATION PASSED
